# Module 3: The Wide Table Pattern — Building a Cohort as a Team
## ESICM LIVES26 — Datathon Educational Series

---

> **Prerequisites:** Modules 1 and 2 completed. You understand the clinical question, how to find concept IDs, and what your cohort's key timestamps represent.
>
> **What this module covers:** A reusable pattern for building and maintaining a shared cohort table as a team — applicable to any future OMOP CDM datathon.

---

### Learning Objectives

By the end of this module, you will be able to:

1. **Design** a schema-first shared cohort table that supports parallel contributions from multiple team members
2. **Implement** idempotent UPDATE patterns that can be run repeatedly without corrupting data
3. **Coordinate** a team data science workflow using a structured file pipeline (create → populate → update → validate)
4. **Apply** post-UPDATE validation checks to catch errors introduced during collaborative feature engineering

> *Bloom's taxonomy level: Synthesis — combining skills from Modules 1 and 2 into a team-level workflow pattern.*

### How to read this module

This notebook is designed for two audiences:

| If you are... | Focus on... |
|---|---|
| **A clinician** | The blockquote boxes at the start of each section — these explain *why* each step exists in clinical terms. You do not need to read the code. |
| **A data analyst or engineer** | Read everything. The blockquotes give you the clinical intent; the code gives you the implementation. |

The Wide Table pattern described here was developed during the 2026 ESICM INDICATE datathon using AmsterdamUMCdb, but the approach works with any OMOP CDM dataset.

### The File Structure

A Wide Table pipeline typically uses a small number of SQL files run in sequence:

```
1. create_cohort.sql        -- DROP + CREATE: declares the full schema (team lead, run once)
2. populate_base.sql        -- base rows: one row per patient, core identifiers and outcomes
3. add_features_A.sql       -- UPDATE: features at landmark A (e.g. eligibility)
4. add_features_B.sql       -- UPDATE: features at landmark B (e.g. intervention)
5. add_derived.sql          -- UPDATE: calculated fields (slopes, ratios, cumulative values)
```

Files 3-5 can be run in any order, by different team members, on the same table.


---
## Setup

In [ ]:
PROJECT_ID         = 'your-datathon-project'
DATASET_PROJECT_ID = 'amsterdamumcdb'
DATASET_ID         = 'van_gogh_2026_datathon_update'
LOCATION           = 'eu'
SANDBOX_PROJECT    = 'your-datathon-project'
SANDBOX_DATASET    = 'sandbox'
COHORT_TABLE       = f'{SANDBOX_PROJECT}.{SANDBOX_DATASET}.patient_weaning'

import os, pandas as pd
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID

job_config = bigquery.job.QueryJobConfig()
def_config  = bigquery.job.QueryJobConfig(
    default_dataset=f'{DATASET_PROJECT_ID}.{DATASET_ID}'
)
client = bigquery.Client(project=PROJECT_ID, location=LOCATION,
                         default_query_job_config=def_config)
print(f'Connected | Working table: {COHORT_TABLE}')

---
## Section 1: Why a Wide Table?

> **Clinical perspective:** Every patient in an ICU has a single shared chart. The physiotherapist writes in it. The nurse writes in it. The intensivist reads from it. Nobody keeps their own separate notes and tries to merge them at the end of the day — because that causes errors.
>
> A Wide Table is the data equivalent: one row per patient, one shared table, every team member adding their columns to it.

### The Problem It Solves

Without a shared table, a typical datathon team ends up doing this:

- Member A runs their query and exports a CSV with 300 rows
- Member B runs their query and exports a different CSV, also 300 rows (they hope)
- Someone tries to merge them at 11pm: column names don't match, row counts differ by 4, nobody knows why
- The model gets built on the merged file, but nobody is confident it is correct

The Wide Table pattern eliminates this. There is one table. Everyone writes to it. At the end, you simply query the table.

### How It Works

```
person_id | core identifiers + outcomes + demographics | feature_A | feature_B | derived
          | <---------- populate_base.sql -----------> | <-- 3+4 --> | <-- 5 --> |
```

One row per patient. All columns declared upfront in the schema. Each file fills its assigned columns. At the end of the datathon, every cell in the table has a value — or a meaningful default that explains why it does not.

### Why We Chose This Pattern

The Wide Table approach was chosen deliberately for the 2026 ESICM INDICATE datathon. Here is why it worked, and why it is worth learning as a reusable pattern:

1. **Speed.** At a datathon, you need results fast. A single flat table with one row per patient lets you start querying immediately — no joins, no assembly. We had demographic statistics on day one, then IMV numbers, then slopes. You do not wait until the last day to show progress.

2. **Scalable development.** The Wide Table supports an incremental workflow: add a few columns, validate, add more. This keeps the team moving and surfaces problems early. For example, if the team could not find half the cohort on mechanical ventilation, something was wrong with the cohort logic — and we caught it before building any models, not after.

3. **Extensible without breaking.** New feature columns get added at the end of the schema. Nobody's existing queries break. Nobody's code needs updating. The table grows sideways, not in complexity.

4. **Everyone understands spreadsheets.** Even a very wide one. At a datathon with mixed clinical and technical backgrounds, this is a significant advantage. The barrier to reading and validating the data is as low as it can be.

5. **Native to BigQuery.** BigQuery is optimised for wide, flat, columnar tables. We were exploiting what the platform gives you natively rather than fighting against it.

6. **Popular format for analytics and ML.** One row per patient with features as columns is the format that visualisation tools, statistical software, and machine learning libraries expect. No reshaping needed.

7. **Feature store analogy.** Over several passes, the team "engineers features" — and the Wide Table accumulates them. Each UPDATE pass adds or refines a set of columns. This is conceptually similar to a feature store in ML engineering, where features are computed independently and joined into a single training table.

> **From proof-of-concept to production:** The Wide Table pattern is ideal for bootstrapping and proof-of-concept work — exactly what a datathon is. Once you have the scent of a result and need to scale, you may branch out into timeseries databases for machine-generated data, or use a blend: keep the Wide Table as a "summary feature metrics" layer and add timeseries storage for high-frequency measurements. But for getting started quickly and iterating as a team, the flat Wide Table is hard to beat.

### Mike Madden's AI/ML Processing

After the team built the Wide Table, Mike Madden took the output and ran it through several machine learning models. The row-per-patient format was something he requested early in the datathon — and the team anticipated this would be needed. Because each row was a self-contained feature vector for one patient, the data plugged directly into his ML pipelines with no additional reshaping. This is one of the practical advantages of the pattern: the output of the data engineering phase is immediately consumable by the modelling phase.

### The Google Connection — Where the Name Comes From

> Google came up with the PageRank algorithm to index the web. To store the results, they built a database called **BigTable** — a single row for each website, with columns added as web crawlers and subsequent processing learned more about each page. Once it had this table — one row per website, growing wider over time — they passed it to their teams to build the search indexes. BigTable columns can store complex objects (documents, nested structures), making it a NoSQL database.
>
> Our pattern is simpler: a flat **Wide Table** native to BigQuery, where each column stores a single value (a number, a timestamp, a string). But the underlying idea is the same — one entity per row, accumulate knowledge as columns. If it worked to index the web, it is not a bad mental model for indexing patient data.
>
> — *Adapted from Tony McNicholas's datathon debrief*


---
## Section 2: The DDL — Declaring the Schema Upfront

> **Clinical perspective:** Before a clinical trial starts, the data collection form is designed. Every variable is listed, its type is defined (number, yes/no, date), and what to record when data is missing is specified. Nobody starts collecting data and then argues about column names halfway through.
>
> The DDL (Data Definition Language) is the equivalent: a `CREATE TABLE` statement that declares every column the team will ever need, before anyone writes a single query.

### The Most Important File

Before anyone writes a feature query, the team lead creates the table schema. This file answers three questions:
- What columns exist?
- What type are they?
- What is the default value while they are being filled?

**Run once. Then never again** — unless the team agrees to add a new column and resets from scratch.

### Why `DROP TABLE` first?

BigQuery will not let you `CREATE TABLE` if the table already exists. At a datathon you may reset the pipeline a few times as you refine your approach. `DROP TABLE IF EXISTS; CREATE TABLE` is the explicit 'start fresh' operation.

> **WARNING:** `DROP TABLE` destroys all data. Only the team lead runs this, and only when the team agrees to reset. Everyone else runs UPDATE files only.


In [ ]:
# Schema declaration -- run ONCE by the team lead before anyone starts.
#
# Design decisions in this schema:
#   - All columns declared upfront -- no surprises later
#   - Feature columns default to NULL: NULL means 'not yet filled by an UPDATE'
#   - Cumulative/running columns default to 0: zero is a valid clinical value
#   - Outcome flags default to 0: assume no event until classified otherwise
#
# Adapt COHORT_TABLE, column names, and types to your datathon's clinical question.

ddl = f"""
DROP TABLE IF EXISTS `{COHORT_TABLE}`;

CREATE TABLE `{COHORT_TABLE}`
(
  -- Core identifiers (filled by populate_base.sql)
  person_id               INT64,
  admission_start         TIMESTAMP,
  admission_end           TIMESTAMP,

  -- Demographics (filled by populate_base.sql)
  age_at_admission        NUMERIC,
  sex                     STRING,

  -- Key clinical timestamps
  -- Define 1-3 'landmarks': clinically meaningful moments for your question
  -- e.g. eligibility for weaning, start of a treatment, day of ICU discharge
  landmark_a_time         TIMESTAMP,
  landmark_b_time         TIMESTAMP,

  -- Feature columns: one per variable, two per landmark (value + timestamp)
  -- Naming convention: feature_name_at_landmark_value / feature_name_at_landmark_time
  -- Add as many as your clinical question requires.
  feature_1_at_a          NUMERIC,
  feature_1_at_a_time     TIMESTAMP,
  feature_1_at_b          NUMERIC,
  feature_1_at_b_time     TIMESTAMP,
  feature_1_slope         NUMERIC,     -- (value_b - value_a) / hours_between

  -- Repeat the block above for each additional feature:
  -- feature_2_at_a, feature_2_at_a_time, feature_2_at_b, feature_2_at_b_time ...

  -- Cumulative values (e.g. fluid balance periods) -- DEFAULT 0, not NULL
  -- Zero is a valid measurement: a patient who received no fluid genuinely has 0 mL
  cumulative_value_period_1   NUMERIC  DEFAULT 0,
  cumulative_value_period_2   NUMERIC  DEFAULT 0,

  -- Outcomes (filled by populate_base.sql)
  outcome_flag            INT64    DEFAULT 0,
  outcome_label           STRING,
  cohort_criteria         INT64,

  -- Housekeeping
  died_timestamp          TIMESTAMP,
  last_known_measurement  TIMESTAMP
)
"""

# Review before running:
# print(ddl)

# To execute:
# client.query(ddl, job_config=job_config).result()
# print('Table created')

print('DDL ready. Adapt column names to your clinical question, then run.')
print('Full schema: create_cohort.sql')


### The `DEFAULT 0` vs `DEFAULT NULL` Decision

> **Clinical perspective:** On a patient's chart, a blank space and a recorded value of zero mean different things. A blank weight field means nobody weighed the patient. A weight of 0 kg is an error. The database default encodes this distinction.

- **`DEFAULT NULL`** — use for feature columns (lab values, vital signs). NULL means 'this UPDATE has not run yet'. You can detect unfilled columns at a glance.
- **`DEFAULT 0`** — use for running totals and cumulative values (fluid balance, event counts). Zero is a clinically valid value: a patient who received no IV fluid in a period genuinely has an intake of 0. Storing NULL would make it impossible to distinguish 'no fluid given' from 'not yet calculated'.

**The default encodes a clinical assumption.** Choosing the wrong default means your SUM queries will produce wrong results, silently.


---
## Section 3: populate_base.sql — One Row Per Patient

> **Clinical perspective:** Before recording any clinical variables, you first need a list of patients who meet the study criteria. This file creates that list — one row per patient — and fills in the core columns that everything else depends on: who the patient is, when they were admitted, whether they met eligibility, and what outcome they had.
>
> Think of this as 'enrolling' patients into the study table.

### Phase 1: TRUNCATE + INSERT

`TRUNCATE` removes all existing rows but keeps the schema intact. `INSERT` then adds one row per eligible patient, filling only the core identifier columns. Everything else stays at DEFAULT.

This two-step approach separates *who is in the cohort* (team lead controls this) from *what we know about them* (everyone fills in their assigned columns).

### Phase 2: Sequential UPDATE chain

After inserting base rows, `populate_base.sql` runs a sequence of UPDATEs to fill the columns it owns. These must run in order, because each step builds on the previous:

| Step | What it does | Why order matters |
|------|-------------|------------------|
| 1 | Define the primary clinical episode (start and end timestamps) | Everything else references this timeline |
| 2 | Identify landmark A timestamp | Depends on the episode being defined |
| 3 | Identify landmark B timestamp | Depends on landmark A |
| 4 | Set cohort inclusion flag | Depends on all timestamps being set |
| 5 | Classify outcome | Depends on cohort flag |
| 6 | Add demographics (age, sex, weight) | Can run after base rows exist |

The exact steps depend on your clinical question. The principle is the same: one chain, one owner (team lead), runs top to bottom.

> **Why is all of this in one file?** These columns form a dependency chain: each step relies on the previous. The team lead owns this logic end-to-end. Splitting it across files would require coordinating run order between team members, defeating the purpose of the parallel-safe design.


In [ ]:
# Base population -- TRUNCATE + INSERT
# Run by the team lead after the DDL.
# Only inserts core identifier columns -- everything else stays at DEFAULT.
#
# Adapt the WHERE clause to your datathon's inclusion/exclusion criteria.

query_truncate = f"""TRUNCATE TABLE `{COHORT_TABLE}`"""
client.query(query_truncate, job_config=job_config).result()
print('Table truncated')

query_insert = f"""
INSERT INTO `{COHORT_TABLE}`
  (person_id, admission_start, admission_end)

-- Select one row per eligible patient.
-- DISTINCT: visit_occurrence may contain duplicate rows in some OMOP implementations.
SELECT DISTINCT
  person_id,
  visit_start_datetime AS admission_start,
  visit_end_datetime   AS admission_end
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.visit_occurrence`
WHERE
  -- Add your inclusion criteria here.
  -- Example: date range, care site type, visit type.
  visit_end_datetime IS NOT NULL
"""
client.query(query_insert, job_config=job_config).result()

# Check how many patients were enrolled
n = client.query(
    f'SELECT COUNT(*) AS n FROM `{COHORT_TABLE}`',
    job_config=job_config
).to_dataframe()
print(f'{n.iloc[0,0]:,} rows inserted')
print('All feature columns are at DEFAULT -- ready to be filled by UPDATE files')


---
## Section 4: Filling Features — The UPDATE Pattern

> **Clinical perspective:** After the patient list exists, each team member adds their assigned clinical variables — vital signs, lab results, medication flags — at the relevant time points. Each person works on their own columns. Nobody overwrites anyone else's work. At the end, the table contains everything.

Every feature UPDATE follows the same structure:

```sql
UPDATE `cohort_table` c
SET
  feature_at_landmark      = m.value,
  feature_at_landmark_time = m.measurement_datetime
FROM (
  SELECT
    m.person_id,
    m.value_as_number AS value,
    m.measurement_datetime
  FROM `cohort_table` c
  JOIN measurement m ON m.person_id = c.person_id
  WHERE
    c.cohort_criteria = 1
    AND m.measurement_concept_id IN (<your concept IDs>)  -- use Module 1 process
    AND m.unit_source_value LIKE '<expected unit>'        -- always filter units
    AND m.measurement_datetime BETWEEN
        DATETIME_SUB(c.landmark_time, INTERVAL 12 HOUR)
        AND c.landmark_time
  -- Pick the measurement closest in time to the landmark
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY m.person_id
    ORDER BY ABS(TIMESTAMP_DIFF(c.landmark_time, m.measurement_datetime, HOUR))
  ) = 1
) m
WHERE c.person_id = m.person_id
```

One UPDATE block per feature per landmark. A file with 10 features at 2 landmarks = 20 UPDATE blocks.

### Why this is safe to run in parallel

Each UPDATE targets specific columns. Two team members running their own feature files simultaneously write to **different columns on the same rows** — there is no collision.

### A Note on Slopes and Time Intervals

If your clinical question involves calculating slopes (rate of change between two landmarks), be aware of two practical problems encountered during the datathon:

1. **Short intervals amplify error.** The slope formula divides the value difference by the time between two measurements. When that interval is narrow — say, two hours — even a small measurement error produces a large slope. You need a long enough interval between the two time points for the slope to be meaningful. During the datathon, this was a pain to manage and required careful filtering.

2. **Measurement gaps are common.** In the datathon dataset, roughly 20% of the cohort had unexplained gaps in measurements that could not be tracked down or explained. This means your slope will be NULL for those patients (which is correct — better NULL than a slope calculated from unreliable data). Plan for this in your analysis: your modelling dataset will be smaller than your cohort for any slope-dependent variable.

The data is close to being clean enough for clear slope-based results, but not quite there yet. Treat slopes as informative features, not as precision instruments.

### Idempotency — re-running is safe

Running the same UPDATE twice produces exactly the same result as running it once. BigQuery UPDATE **overwrites** — it does not accumulate. This means:

- Re-run after a kernel restart: safe
- Re-run after fixing a unit filter: safe
- A colleague re-runs your file to check it: safe

This property is called *idempotency*. Write your UPDATEs so that running them twice is harmless.


In [ ]:
# Demonstration: a single UPDATE block -- the fundamental unit of the Wide Table pattern.
#
# This example fills feature_1 (e.g. heart rate, systolic BP, haemoglobin)
# at landmark B (e.g. the intervention or weaning time) for all cohort patients.
#
# To adapt this for your datathon:
#   1. Replace feature_1_at_b / feature_1_at_b_time with your column names
#   2. Replace XXXXXXXX / YYYYYYYY with your validated concept IDs (Module 1 process)
#   3. Replace 'expected_unit' with the unit string from your unit validation (Module 2)
#   4. Replace landmark_b_time with your landmark timestamp column
#   5. Adjust the time window (INTERVAL 12 HOUR) to match your clinical question

query_feature = f"""
UPDATE `{COHORT_TABLE}` c
SET
  feature_1_at_b      = ROUND(CAST(m.value AS NUMERIC), 4),
  feature_1_at_b_time = m.measurement_datetime
FROM (
  SELECT
    m.person_id,
    m.value_as_number AS value,
    m.measurement_datetime
  FROM `{COHORT_TABLE}` c
  JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
    ON m.person_id = c.person_id
  WHERE
    c.cohort_criteria = 1
    -- Step 1: filter to your concept IDs (validate using Module 1 process)
    AND m.measurement_concept_id IN (XXXXXXXX, YYYYYYYY)
    -- Step 2: filter to the expected unit (see Module 2 for unit validation)
    AND m.unit_source_value LIKE 'expected_unit'
    -- Step 3: restrict to a time window around the clinical landmark
    AND m.measurement_datetime BETWEEN
        DATETIME_SUB(c.landmark_b_time, INTERVAL 12 HOUR)
        AND c.landmark_b_time
  -- Step 4: pick the measurement closest in time to the landmark
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY m.person_id
    ORDER BY ABS(TIMESTAMP_DIFF(c.landmark_b_time, m.measurement_datetime, HOUR))
  ) = 1
) m
WHERE c.person_id = m.person_id
"""

# Uncomment to run:
# client.query(query_feature, job_config=job_config).result()
# print('feature_1_at_b filled')
# print('Safe to re-run -- UPDATE overwrites, never accumulates')

print('Adapt concept IDs, unit, landmark, and column names -- then run.')
print('Repeat one block per feature per landmark.')


---
## Section 5: Validation — Checking Your Work After Every UPDATE

> **Clinical perspective:** A lab result transcribed with the wrong unit — mmol/L instead of mg/dL — looks like a valid number. The column is populated. There is no error message. But the model will train on values that are 18 times too high.
>
> This is why you validate immediately after filling each column, not at the end when it is too late to easily identify which file caused the error.

The validation pattern checks two things after every UPDATE:

1. **NULL percentage** — very high NULL% suggests wrong concept IDs or a unit filter that matched nothing
2. **Value range** — values outside the expected clinical range suggest a unit error

This was not part of the original datathon pipeline — the source SQL files contain no validation layer. At a fast-moving datathon, you run the UPDATE and move on. This is a gap. A 5-minute validation step can prevent hours of debugging during model building.


In [ ]:
# Validation helper -- run this after every UPDATE in your pipeline.
#
# check_column() reports: NULL%, mean, min, max vs your expected clinical range.
# If values are outside the expected range: wrong unit filter or wrong concept IDs.

def check_column(client, table, column, expected_min, expected_max, job_config,
                 where_clause='cohort_criteria = 1'):
    """Quick sanity check: NULL%, mean, min, max vs expected clinical range."""
    row = client.query(f"""
      SELECT
        COUNTIF({column} IS NULL) * 100.0 / COUNT(*) AS pct_null,
        ROUND(AVG({column}), 3)  AS mean_val,
        MIN({column})            AS min_val,
        MAX({column})            AS max_val
      FROM `{table}` WHERE {where_clause}
    """, job_config=job_config).to_dataframe().iloc[0]

    in_range = (row['min_val'] is not None
                and row['min_val'] >= expected_min
                and row['max_val'] <= expected_max)
    status = 'PASS' if in_range else 'FAIL - check units / concept IDs'
    print(f'{column}: NULL={row["pct_null"]:.1f}% | mean={row["mean_val"]} | '
          f'range={row["min_val"]}--{row["max_val"]} -> {status}')


# Define expected clinical ranges for your features.
# These catch the most common errors at datathons:
#   - Wrong unit (e.g. FiO2 stored as 0.21 fraction instead of 21%)
#   - Wrong concept ID (e.g. SpO2 mixed up with FiO2)
#   - Implausible value (e.g. negative creatinine, impossible HR)
#
# Adapt these ranges to match your variables and expected units.
EXPECTED_RANGES = {
    # Vital signs
    'heart_rate_at_a':        (20, 250),   # bpm
    'systolic_bp_at_a':       (50, 300),   # mmHg -- if >400: stored in Pa
    'spo2_at_a':              (50, 100),   # %
    # Respiratory
    'fio2_at_a':              (21, 100),   # % -- if 0.21 to 1.0: stored as fraction
    'peep_at_a':              (0, 30),     # cmH2O
    # Labs (adjust for your units)
    'haemoglobin_at_a':       (2, 20),     # mmol/L -- if >50: stored as g/dL
    'creatinine_at_a':        (20, 2000),  # umol/L
    'lactate_at_a':           (0, 30),     # mmol/L
    # Cumulative values (DEFAULT 0 so never NULL)
    'cumulative_value_period_1': (-5000, 10000),  # e.g. fluid balance in mL
}

# Run after each UPDATE. Replace column names with your actual schema.
# Example:
for col, (lo, hi) in list(EXPECTED_RANGES.items())[:3]:
    print(f'  check_column(client, COHORT_TABLE, "{col}", {lo}, {hi}, job_config)')
print('  ...')
print('Adapt column names and ranges to your schema, then uncomment and run.')


---
## Section 6: Team Coordination

> **Clinical perspective:** In a clinical trial, the data manager assigns which data collector fills which section of the CRF. Nobody fills the same field twice. Nobody leaves a field empty by accident. This section describes the same principle for a datathon team.

### Dividing the Work

The file structure is the division of labour. Assign ownership before writing any code:

| File | Owner | What it does |
|------|-------|--------------|
| `create_cohort.sql` | Team lead | Schema -- run once |
| `populate_base.sql` | Team lead | Cohort logic, outcomes, demographics |
| `add_features_A.sql` | Member A | Features at landmark A |
| `add_features_B.sql` | Member B | Features at landmark B |
| `add_derived.sql` | Any | Slopes, ratios (runs last) |

The file boundary is the ownership boundary. Each person runs their file, validates their columns, and commits their code. The shared table is the integration point.

### Column Registry

Create a shared document (a Google Sheet works well) before writing any code:

| Column | File | Owner | Status | Validated |
|--------|------|-------|--------|-----------|
| heart_rate_at_a | add_features_A.sql | Member A | Done | Done |
| systolic_bp_at_a | add_features_A.sql | Member A | Done | Done |
| haemoglobin_at_b | add_features_B.sql | Member B | In progress | -- |

This prevents two people working on the same column and ensures nothing is left empty at submission.

### Recovery: Re-running After a Bug

If an UPDATE was wrong (wrong unit filter, wrong time window), run the corrected version again. It overwrites. No need to drop the table or start over:

```sql
-- Fix: added missing unit filter
UPDATE cohort_table SET feature_1_at_b = ...
WHERE ...
AND m.unit_source_value LIKE 'correct_unit'  -- this was missing before
```


In [ ]:
# Table population status — run at the start of each session
# Shows which numeric columns are filled vs. still NULL

def table_status(client, table, job_config):
    parts = table.split('.')
    cols = client.query(f"""
      SELECT column_name, data_type
      FROM `{parts[0]}.{parts[1]}.INFORMATION_SCHEMA.COLUMNS`
      WHERE table_name = '{parts[2]}'
      AND data_type IN ('NUMERIC','INT64','FLOAT64')
      ORDER BY ordinal_position
    """, job_config=job_config).to_dataframe()

    rows = []
    for col in cols['column_name']:
        pct = client.query(f"""
          SELECT ROUND(COUNTIF({col} IS NOT NULL) * 100.0 / COUNT(*), 0) AS pct
          FROM `{table}` WHERE cohort_criteria = 1
        """, job_config=job_config).to_dataframe().iloc[0, 0]
        rows.append({'column': col,
                     'pct_populated': pct or 0,
                     'status': 'OK' if (pct or 0) >= 80 else ('partial' if (pct or 0) > 0 else 'empty')})

    df = pd.DataFrame(rows)
    print('=== Table Status ===')
    print(df.to_string(index=False))
    return df

# status = table_status(client, COHORT_TABLE, job_config)

---
## Section 7: Designing Your Column Map

> **Clinical perspective:** Before starting data collection in any study, you need a data dictionary: what variable is being collected, what it means, what unit it is in, and who is responsible for it. The column map is that dictionary for your Wide Table.

A well-designed column map, created before the datathon starts, will save your team significant time. Here is a template:

### Group 1: Core identifiers
`person_id`, `admission_start`, `admission_end`

### Group 2: Clinical episode
Define the primary clinical process you are studying:
- Start and end timestamps for the episode (e.g. ventilation, a treatment, ICU stay)
- Duration calculations derived from those timestamps

### Group 3: Landmarks (1-3 time points)
Define clinically meaningful moments:
- When does the patient become eligible for the event of interest?
- When does the intervention or event occur?
- Are any follow-up time points needed?

### Group 4: Features x landmarks
For each clinical variable and each landmark, you need:
- The value: `feature_name_at_landmark`
- The measurement timestamp: `feature_name_at_landmark_time`
- Optionally the slope between landmarks: `feature_name_slope`

**How many features?** At a 48-hour datathon, 8-15 features across 2 landmarks is realistic. Each feature requires one UPDATE block (~15 lines of SQL). Plan the column registry before writing any code.

### Group 5: Running totals (if applicable)
For cumulative variables (fluid balance, medication doses), define periods that match your clinical question:
- Period 1: from episode start to landmark A
- Period 2: from landmark A to landmark B
- Period 3: last N hours before landmark B

### Group 6: Outcomes
At minimum:
- `outcome_flag` (INT64, DEFAULT 0): primary binary outcome
- `outcome_label` (STRING): human-readable classification
- `cohort_criteria` (INT64): 1 = in final cohort, NULL = excluded

### Group 7: Housekeeping
- `died_timestamp`: death during admission
- `last_known_measurement`: proxy for last-known-alive
- Any re-admission, transfer, or census flags relevant to your question


## Checklist: Building a Wide Table for Your Datathon

### Before the datathon
- Define your clinical question and the 1-3 key timestamps (landmarks)
- List all features you need, grouped by landmark
- Write the DDL: declare every column, type, and default
- Create the column registry: assign each column to a team member
- Validate your concept IDs for each feature (Module 1 process)

### Day 1 (team lead)
- Run DDL: `DROP TABLE IF EXISTS` + `CREATE TABLE`
- Run base population: `TRUNCATE` + `INSERT` core columns
- Share `COHORT_TABLE` variable and registry with everyone
- Run the Phase 2 UPDATE chain in `populate_base.sql`

### Each session (per team member)
- Run your UPDATE file for your assigned columns
- Run `check_column()` on each column you just filled
- Update the column registry: mark your columns done
- If NULL% is high or values are out of range: revisit concept IDs and unit filters

### Before analysis
- Run `table_status()` -- confirm all columns are populated
- Run the full validation sweep with `EXPECTED_RANGES`
- Lock: no more UPDATEs without team agreement
- Export or query directly from `COHORT_TABLE` for modelling


---
## The Pattern -- Summary

```
1. DDL (team lead, once)
   DROP TABLE + CREATE TABLE with ALL columns declared
   Types + DEFAULT values encode clinical assumptions about missing data

2. populate_base.sql (team lead, after DDL)
   TRUNCATE + INSERT: base rows (core identifiers only)
   Then sequential UPDATEs -- must run in order:
     clinical episode -> landmarks -> cohort criteria -> outcomes -> demographics

3-5. Feature files (per domain, any order, parallel-safe)
   UPDATE cohort_table SET feature = subquery
   Idempotent: re-running overwrites, never accumulates
   Each file owns its columns -- no conflicts between team members

6. Validate (after every UPDATE)
   check_column(feature, expected_min, expected_max)
   NULL% + range check catches unit errors before they reach the model
```

### Why This Works

| Property | How | Why it matters |
|----------|-----|----------------|
| Full schema upfront | DDL declares all columns before data collection | No surprises; everyone knows the contract |
| DEFAULT encodes clinical meaning | NULL = not filled; 0 = valid zero value | Safe SUM and AVG queries from the start |
| Sequential UPDATEs in populate_base | Dependency chain in one file | Each step builds on the previous; one owner |
| Idempotent UPDATEs in feature files | Overwrite, never accumulate | Safe to re-run, safe to fix bugs |
| File = ownership | One file per domain | No column conflicts between team members |
| Validate after each UPDATE | check_column() | Catches unit errors before analysis |

### Transferability

This pattern works with any OMOP CDM dataset and any clinical question. The specific SQL in your feature files will change -- the structure will not. Start your next datathon by writing the DDL and column registry before any code is written.

---

*ESICM LIVES26 Educational Series — Datathon Track*
*Developed during the 2026 ESICM INDICATE datathon. Pattern applicable to any OMOP CDM datathon.*

---

### Source Code & Notebooks

All modules in this series are available at:

**[github.com/ReginaldCal/Digital-Critical-Care-Datathon-Prep](https://github.com/ReginaldCal/Digital-Critical-Care-Datathon-Prep)**
